# Heard — Data Cleaning for Training

**Primary source**
- Kaggle: `davidgauthier/glassdoor-job-reviews`
- Role: historical labeled data for training / validation

**Supplementary source**
- Bright Data / GitHub: `luminati-io/Glassdoor-dataset-samples`
- Role: recent external evaluation and RAG/demo data
- We keep reviews dated **2022 or later** so they supplement the Kaggle dataset, which ends in 2021.

## Scope

This notebook performs **data cleaning and training preparation**:

1. Load both datasets.
2. Standardize variable names.
3. Parse dates and ratings.
4. Normalize categorical values.
5. Clean review text.
6. Remove unusable and duplicate rows.
7. Create a common `review_text` field.
8. Create simple sentiment labels from numeric ratings.
9. Save cleaned training and external-evaluation files.

> The Bright Data sample is intentionally kept separate from the Kaggle training data to reduce leakage and provide an out-of-time external evaluation set.



## 1. Setup

Download the Kaggle CSV and place it inside:

`data/raw/`

The notebook identifies the Kaggle file automatically by checking for expected columns, so the exact CSV filename is not important.

For Bright Data, the notebook first looks for:

`data/raw/glassdoor-companies-reviews.csv`

If it is not present, it attempts to read the public raw GitHub CSV directly.


In [ ]:

from pathlib import Path
import html
import re
import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents = True, exist_ok = True)
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

BRIGHT_LOCAL_PATH = RAW_DIR / "glassdoor-companies-reviews.csv"
BRIGHT_RAW_URL = (
    "https://raw.githubusercontent.com/"
    "luminati-io/Glassdoor-dataset-samples/main/"
    "glassdoor-companies-reviews.csv"
)

print("Raw data folder:", RAW_DIR.resolve())
print("Processed data folder:", PROCESSED_DIR.resolve())



## 2. Load the raw datasets

### Kaggle expected fields

The Kaggle source is expected to contain fields such as:

`firm`, `date_review`, `job_title`, `current`, `location`, `overall_rating`,
`work_life_balance`, `culture_values`, `diversity_inclusion`, `career_opp`,
`comp_benefits`, `senior_mgmt`, `recommend`, `ceo_approv`, `outlook`,
`headline`, `pros`, and `cons`.

### Bright Data / GitHub field mapping

The Bright Data sample uses different names for equivalent concepts:

- `company_name` → `firm`
- `rating_date` → `date_review`
- `review_pros` → `pros`
- `review_cons` → `cons`
- `rating_work_life` → `work_life_balance`
- `rating_compensation_benefits` → `comp_benefits`
- `rating_culture_values` → `culture_values`
- `rating_career_opportunities` → `career_opp`
- `rating_senior_leadership` → `senior_mgmt`

### Review IDs

- Kaggle: assigned `kaggle_000001`, etc. **before cleaning and splitting**;
  IDs remain unchanged when you rerun or reshuffle splits from the same
  unchanged source CSV (including its row order).
- Bright Data: preserve source `review_id` when provided; otherwise create a
  deterministic `bright_...` ID from the original review fields. IDs are
  carried through all downstream sentiment/aspect exports.


In [ ]:

KAGGLE_REQUIRED = {
    "firm",
    "date_review",
    "overall_rating",
    "pros",
    "cons",
}

def find_kaggle_csv(search_dirs = (RAW_DIR, Path("."))):
    candidates = []
    for folder in search_dirs:
        if folder.exists():
            candidates.extend(folder.glob("*.csv"))

    for path in candidates:
        if path.name == BRIGHT_LOCAL_PATH.name:
            continue

        try:
            sample = pd.read_csv(path, nrows = 5, low_memory = False)
        except Exception:
            continue

        if KAGGLE_REQUIRED.issubset(sample.columns):
            return path

    raise FileNotFoundError(
        "Kaggle CSV not found. Download the Kaggle Glassdoor Job Reviews dataset "
        "and place its CSV file inside data/raw/."
    )

kaggle_path = find_kaggle_csv()
kaggle_raw = pd.read_csv(kaggle_path, low_memory=False)

# Assign Kaggle IDs ONCE, before cleaning or splitting.
# Stable across reruns/splits as long as the original CSV row order does not change.
kaggle_raw["review_id"] = [
    f"kaggle_{i:06d}"
    for i in range(1, len(kaggle_raw) + 1)
]

print("Kaggle file:", kaggle_path)
print("Kaggle shape:", kaggle_raw.shape)
display(kaggle_raw.head(3))

In [ ]:

def load_bright_data():
    if BRIGHT_LOCAL_PATH.exists():
        print("Loading local Bright Data file:", BRIGHT_LOCAL_PATH)
        return pd.read_csv(BRIGHT_LOCAL_PATH, low_memory = False)

    print("Local Bright Data file not found; reading public GitHub CSV.")
    try:
        return pd.read_csv(BRIGHT_RAW_URL, low_memory = False)
    except Exception as exc:
        raise RuntimeError(
            "Could not load the Bright Data CSV from GitHub. "
            "Download glassdoor-companies-reviews.csv from the repository "
            "and place it inside data/raw/."
        ) from exc

bright_raw = load_bright_data()

print("Bright Data shape:", bright_raw.shape)
display(bright_raw.head(3))



## 3. Inspect the raw schemas

Makes source changes visible before cleaning. Optional missing columns will be added as `NaN` later so both datasets can still be transformed into one common schema.


In [ ]:

print("KAGGLE COLUMNS")
print(kaggle_raw.columns.tolist())

print("\nBRIGHT DATA COLUMNS")
print(bright_raw.columns.tolist())



## 4. Define the common schema

The common schema keeps fields useful for:

- text classification;
- sentiment prediction;
- aspect-level prediction;
- external evaluation;
- later RAG retrieval.

The Bright Data field `timestamp` is **not** used as the review date. We use `rating_date`, which represents when the review was submitted.


In [ ]:

COMMON_COLUMNS = [
    "firm",
    "date_review",
    "job_title",
    "current",
    "location",
    "overall_rating",
    "work_life_balance",
    "culture_values",
    "diversity_inclusion",
    "career_opp",
    "comp_benefits",
    "senior_mgmt",
    "recommend",
    "ceo_approv",
    "outlook",
    "headline",
    "pros",
    "cons",
    "review_id",
    "review_url",
    "source",
]

ASPECT_COLUMNS = [
    "work_life_balance",
    "culture_values",
    "diversity_inclusion",
    "career_opp",
    "comp_benefits",
    "senior_mgmt",
]

BRIGHT_RENAME = {
    "company_name": "firm",
    "rating_date": "date_review",
    "employee_job_title": "job_title",
    "employee_type": "current",
    "employee_location": "location",
    "rating_overall": "overall_rating",
    "rating_work_life": "work_life_balance",
    "rating_culture_values": "culture_values",
    "rating_diversity_inclusion": "diversity_inclusion",
    "rating_career_opportunities": "career_opp",
    "rating_compensation_benefits": "comp_benefits",
    "rating_senior_leadership": "senior_mgmt",
    "flags_recommend_frend": "recommend",
    "flags_ceo_approval": "ceo_approv",
    "flags_business_outlook": "outlook",
    "summary": "headline",
    "review_pros": "pros",
    "review_cons": "cons",
}



## 5. Cleaning helpers

For transformer / LLM training, text is cleaned **lightly**.

We:
- decode HTML entities such as `&amp;`;
- remove HTML tags;
- remove embedded URLs;
- normalize whitespace;
- preserve normal casing, punctuation, and wording.

We do **not** stem, lemmatize, remove stopwords, or strip punctuation because modern language models benefit from retaining the original language structure.


In [ ]:

NULL_STRINGS = {
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "unknown",
}

def clean_text(value):
    if pd.isna(value):
        return np.nan

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if text.lower() in NULL_STRINGS:
        return np.nan

    return text if text else np.nan


def clean_rating(series):
    # Convert ratings to numeric and keep only the valid 1-5 scale.
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where(numeric.between(1, 5))


def normalize_token(value):
    if pd.isna(value):
        return np.nan

    token = str(value).strip().lower()
    token = re.sub(r"[\s\-]+", "_", token)

    if token in NULL_STRINGS:
        return np.nan

    return token


def normalize_recommend(value):
    token = normalize_token(value)
    if pd.isna(token):
        return np.nan

    mapping = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "no_opinion": "no_opinion",
        "v": "positive",
        "r": "neutral",
        "x": "negative",
        "o": "no_opinion",
        "recommend": "positive",
        "recommended": "positive",
        "not_recommend": "negative",
        "not_recommended": "negative",
        "true": "positive",
        "false": "negative",
        "1": "positive",
        "0": "negative",
    }
    return mapping.get(token, token)


def normalize_ceo(value):
    token = normalize_token(value)
    if pd.isna(token):
        return np.nan

    mapping = {
        "approve": "positive",
        "approved": "positive",
        "positive": "positive",
        "disapprove": "negative",
        "disapproved": "negative",
        "negative": "negative",
        "neutral": "neutral",
        "no_opinion": "no_opinion",
        "v": "positive",
        "r": "neutral",
        "x": "negative",
        "o": "no_opinion",
    }
    return mapping.get(token, token)


def normalize_outlook(value):
    token = normalize_token(value)
    if pd.isna(token):
        return np.nan

    mapping = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "no_opinion": "no_opinion",
        "v": "positive",
        "r": "neutral",
        "x": "negative",
        "o": "no_opinion",
    }
    return mapping.get(token, token)


def rating_to_sentiment(value):
    # Derived target: 1-2 negative, 3 neutral, 4-5 positive.
    if pd.isna(value):
        return np.nan
    if value <= 2:
        return "negative"
    if value == 3:
        return "neutral"
    return "positive"


def ensure_columns(df, columns):
    out = df.copy()
    for column in columns:
        if column not in out.columns:
            out[column] = np.nan
    return out



## 6. Clean the Kaggle dataset

The Kaggle source remains the **primary training source**.

Cleaning steps:

- parse review dates;
- keep valid ratings from 1 to 5;
- normalize categorical variables;
- lightly clean text;
- combine title, pros, and cons into one `review_text`;
- remove unusable short reviews;
- remove duplicate reviews.


In [ ]:

def clean_common_fields(df, source_name):
    out = ensure_columns(df, COMMON_COLUMNS).copy()
    out["source"] = source_name

    out["date_review"] = (out["date_review"].astype(str).str.replace('"', '', regex = False).str.strip())
    out["date_review"] = pd.to_datetime(out["date_review"], errors = "coerce", utc = True)

    rating_columns = ["overall_rating"] + ASPECT_COLUMNS
    for column in rating_columns:
        out[column] = clean_rating(out[column])

    text_columns = [
        "firm",
        "job_title",
        "current",
        "location",
        "headline",
        "pros",
        "cons",
        "review_id",
        "review_url",
    ]

    for column in text_columns:
        out[column] = out[column].map(clean_text)

    out["recommend"] = out["recommend"].map(normalize_recommend)
    out["ceo_approv"] = out["ceo_approv"].map(normalize_ceo)
    out["outlook"] = out["outlook"].map(normalize_outlook)

    def combine_review_text(row):
        parts = []

        if pd.notna(row["headline"]):
            parts.append(f"TITLE: {row['headline']}")
        if pd.notna(row["pros"]):
            parts.append(f"PROS: {row['pros']}")
        if pd.notna(row["cons"]):
            parts.append(f"CONS: {row['cons']}")

        text = " ".join(parts).strip()
        return text if text else np.nan

    out["review_text"] = out.apply(combine_review_text, axis = 1)
    out["text_length"] = out["review_text"].fillna("").str.len()

    out["sentiment_label"] = out["overall_rating"].map(
        rating_to_sentiment
    )

    out = out[out["review_text"].notna()].copy()
    out = out[out["text_length"] >= 20].copy()

    return out


kaggle_clean = clean_common_fields(
    kaggle_raw,
    source_name = "kaggle_glassdoor",
)

before = len(kaggle_clean)

kaggle_clean = kaggle_clean.drop_duplicates(
    subset = ["firm", "date_review", "review_text"],
    keep = "first",
).reset_index(drop = True)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", len(kaggle_clean))
print(
    "Date range:",
    kaggle_clean["date_review"].min(),
    "to",
    kaggle_clean["date_review"].max(),
)

display(kaggle_clean.head(3))



## 7. Clean and standardize Bright Data

The Bright Data source is first renamed into the common Kaggle-style schema.

The same cleaning rules are then applied so both datasets are processed consistently.


In [ ]:

bright_standardized = bright_raw.rename(
    columns = BRIGHT_RENAME
).copy()

# Preserve a genuine source review ID when available. For missing IDs, create a
# deterministic ID from the ORIGINAL Bright Data review fields. Unlike row-based
# numbering, the same source review keeps its ID if the CSV is reordered.
BRIGHT_ID_FIELDS = [
    "firm", "date_review", "job_title", "location",
    "headline", "pros", "cons",
]

def make_bright_review_id(row):
    values = [
        str(row[col]).strip() if pd.notna(row[col]) else ""
        for col in BRIGHT_ID_FIELDS
    ]
    digest = hashlib.sha256("\x1f".join(values).encode("utf-8")).hexdigest()[:16]
    return f"bright_{digest}"

if "review_id" not in bright_standardized.columns:
    bright_standardized["review_id"] = pd.NA

missing_bright_ids = (
    bright_standardized["review_id"].isna()
    | bright_standardized["review_id"].astype("string").str.strip().eq("").fillna(True)
)
bright_standardized.loc[missing_bright_ids, "review_id"] = (
    bright_standardized.loc[missing_bright_ids]
    .apply(make_bright_review_id, axis=1)
)

bright_clean = clean_common_fields(
    bright_standardized,
    source_name = "brightdata_glassdoor",
)

before = len(bright_clean)

bright_clean = bright_clean.drop_duplicates(
    subset = ["firm", "date_review", "review_text"],
    keep = "first",
).reset_index(drop = True)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", len(bright_clean))
print(
    "Full Bright Data review-date range:",
    bright_clean["date_review"].min(),
    "to",
    bright_clean["date_review"].max(),
)

display(bright_clean.head(3))



## 8. Keep recent Bright Data reviews

To make the supplementary source useful as an out-of-time dataset, keep only reviews dated **January 1, 2022 or later**.

This reduces temporal overlap with the Kaggle source and better represents recent employee feedback.


In [ ]:

RECENT_START = pd.Timestamp("2022-01-01", tz="UTC")

bright_recent = bright_clean[
    bright_clean["date_review"].notna()
    & (bright_clean["date_review"] >= RECENT_START)
].copy()

bright_recent = bright_recent.reset_index(drop = True)

print("Bright Data rows before recent-date filter:", len(bright_clean))
print("Bright Data rows from 2022 onward:", len(bright_recent))

if len(bright_recent) > 0:
    print(
        "Recent review-date range:",
        bright_recent["date_review"].min(),
        "to",
        bright_recent["date_review"].max(),
    )

display(bright_recent.head(3))



## 9. Cross-source duplicate check

Even after filtering by date, check whether normalized review text appears in both sources.

Any matching row is removed from the Bright Data external set to reduce leakage.


In [ ]:

def normalize_for_duplicate_check(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower()
    text = re.sub(r"\W+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


kaggle_text_keys = set(
    kaggle_clean["review_text"]
    .map(normalize_for_duplicate_check)
    .dropna()
)

bright_recent["duplicate_key"] = bright_recent["review_text"].map(
    normalize_for_duplicate_check
)

cross_source_duplicate_mask = bright_recent["duplicate_key"].isin(
    kaggle_text_keys
)

print(
    "Cross-source duplicate rows found:",
    int(cross_source_duplicate_mask.sum()),
)

bright_recent = (
    bright_recent.loc[~cross_source_duplicate_mask]
    .drop(columns="duplicate_key")
    .reset_index(drop=True)
)



## 10. Prepare training-ready table functions

This section defines reusable functions and creates full cleaned tables for exploration. The actual Kaggle train/validation/test datasets are created later at the **review level** so that one review cannot leak across splits. Each task-ready table retains `review_id` so model predictions and RAG evidence can be joined to the original review.


### Overall sentiment

The notebook creates a simple derived sentiment label from `overall_rating`:

- 1–2 → `negative`
- 3 → `neutral`
- 4–5 → `positive`

### Aspect-level data

For every non-missing aspect rating, one row is created containing:

- `review_text`
- `aspect`
- `aspect_rating`
- `aspect_sentiment`
- `task_text`

This supports aspect-sentiment training.

> Important: these aspect ratings are **not true theme-presence labels**. A rating tells us how the employee scored a dimension, but it does not prove that the review text explicitly discusses that theme.


In [ ]:

def make_sentiment_table(df):
    columns = [
        "review_id",
        "source",
        "firm",
        "date_review",
        "job_title",
        "location",
        "review_text",
        "overall_rating",
        "sentiment_label",
    ]

    out = df[columns].copy()
    out = out.dropna(
        subset=["review_text", "sentiment_label"]
    )

    return out.reset_index(drop=True)


def make_aspect_table(df):
    records = []

    meta_columns = [
        "review_id",
        "source",
        "firm",
        "date_review",
        "job_title",
        "location",
        "review_text",
    ]

    for aspect in ASPECT_COLUMNS:
        subset = df.loc[
            df[aspect].notna(),
            meta_columns + [aspect],
        ].copy()

        subset = subset.rename(
            columns={aspect: "aspect_rating"}
        )

        subset["aspect"] = aspect

        subset["aspect_sentiment"] = (
            subset["aspect_rating"]
            .map(rating_to_sentiment)
        )

        subset["task_text"] = (
            "ASPECT: "
            + subset["aspect"]
            + " REVIEW: "
            + subset["review_text"]
        )

        records.append(subset)

    if not records:
        return pd.DataFrame()

    out = pd.concat(records, ignore_index=True)

    return out[
        meta_columns
        + [
            "aspect",
            "aspect_rating",
            "aspect_sentiment",
            "task_text",
        ]
    ]


kaggle_sentiment_all = make_sentiment_table(
    kaggle_clean
)

kaggle_aspect_all = make_aspect_table(
    kaggle_clean
)

bright_sentiment_eval = make_sentiment_table(
    bright_recent
)

bright_aspect_eval = make_aspect_table(
    bright_recent
)

print(
    "Kaggle overall-sentiment rows (all cleaned data):",
    len(kaggle_sentiment_all),
)

print(
    "Kaggle aspect rows (all cleaned data):",
    len(kaggle_aspect_all),
)

print(
    "Bright recent overall-sentiment rows:",
    len(bright_sentiment_eval),
)

print(
    "Bright recent aspect rows:",
    len(bright_aspect_eval),
)



## 11. Explore the cleaned datasets

Before splitting the Kaggle data, we inspect the two cleaned sources to understand their size, date coverage, missingness, label balance, and text length.

The goal is not a full EDA. These checks focus on variables that matter for model training and external evaluation:

- number of usable reviews;
- date range;
- overall rating and sentiment distribution;
- missingness in the six aspect ratings;
- review-text length;
- review counts by year.

The Bright Data dataset remains separate and represents the newer **2022–2024 external evaluation source**.


In [ ]:

def basic_dataset_summary(name, df):
    return {
        "dataset": name,
        "rows": len(df),
        "firms": df["firm"].nunique(dropna=True),
        "start_date": df["date_review"].min(),
        "end_date": df["date_review"].max(),
        "avg_overall_rating": df["overall_rating"].mean(),
        "median_text_length": df["text_length"].median(),
        "avg_text_length": df["text_length"].mean(),
    }


dataset_summary = pd.DataFrame([
    basic_dataset_summary("Kaggle cleaned", kaggle_clean),
    basic_dataset_summary("Bright Data 2022+", bright_recent),
])

dataset_summary["avg_overall_rating"] = (
    dataset_summary["avg_overall_rating"].round(2)
)
dataset_summary["median_text_length"] = (
    dataset_summary["median_text_length"].round(1)
)
dataset_summary["avg_text_length"] = (
    dataset_summary["avg_text_length"].round(1)
)

display(dataset_summary)


In [ ]:

# Compare missingness in the rating variables.

rating_columns = ["overall_rating"] + ASPECT_COLUMNS

missingness_comparison = pd.DataFrame({
    "Kaggle_missing_%": (
        kaggle_clean[rating_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
    "Bright_2022plus_missing_%": (
        bright_recent[rating_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
})

display(missingness_comparison)


In [ ]:

# Compare overall sentiment distributions.

kaggle_sentiment_distribution = (
    kaggle_sentiment_all["sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("Kaggle_%")
)

bright_sentiment_distribution = (
    bright_sentiment_eval["sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("Bright_2022plus_%")
)

sentiment_comparison = pd.concat(
    [
        kaggle_sentiment_distribution,
        bright_sentiment_distribution,
    ],
    axis=1,
).fillna(0)

display(sentiment_comparison)


In [ ]:

# Review counts by year.

kaggle_reviews_by_year = (
    kaggle_clean
    .assign(year=kaggle_clean["date_review"].dt.year)
    .groupby("year")
    .size()
    .rename("Kaggle_reviews")
)

bright_reviews_by_year = (
    bright_recent
    .assign(year=bright_recent["date_review"].dt.year)
    .groupby("year")
    .size()
    .rename("Bright_reviews")
)

reviews_by_year = pd.concat(
    [kaggle_reviews_by_year, bright_reviews_by_year],
    axis=1,
).fillna(0).astype(int)

display(reviews_by_year)


In [ ]:

# Review-text length summary.

text_length_summary = pd.DataFrame({
    "Kaggle": kaggle_clean["text_length"].describe(),
    "Bright_2022plus": bright_recent["text_length"].describe(),
}).round(2)

display(text_length_summary)



## 12. Define training, validation, and test datasets

We split the **Kaggle cleaned reviews** into:

- **80% training set** — used to fine-tune the model;
- **10% validation set** — used for model selection, hyperparameter decisions, and monitoring overfitting;
- **10% internal test set** — used once for final evaluation on unseen reviews from the same Kaggle source.

The **Bright Data 2022+ reviews are not included in this split**. They remain a completely separate **external test set** for checking whether a model trained on historical Kaggle reviews generalizes to newer employee feedback.

The split is performed at the **review level before creating aspect-level rows**. This prevents the same review from appearing in multiple splits through different aspect ratings.

We stratify by `sentiment_label` so that the overall positive / neutral / negative proportions remain similar across Kaggle train, validation, and internal test sets.


In [ ]:

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Use reviews with an available overall sentiment label for supervised splitting.
kaggle_split_base = (
    kaggle_clean
    .dropna(subset=["review_text", "sentiment_label"])
    .copy()
)

# First split: 80% training, 20% temporary holdout.
kaggle_train_reviews, kaggle_holdout_reviews = train_test_split(
    kaggle_split_base,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=kaggle_split_base["sentiment_label"],
)

# Second split: divide the 20% holdout equally into
# 10% validation and 10% internal test.
kaggle_validation_reviews, kaggle_test_reviews = train_test_split(
    kaggle_holdout_reviews,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=kaggle_holdout_reviews["sentiment_label"],
)

# Reset indices after splitting.
kaggle_train_reviews = kaggle_train_reviews.reset_index(drop=True)
kaggle_validation_reviews = kaggle_validation_reviews.reset_index(drop=True)
kaggle_test_reviews = kaggle_test_reviews.reset_index(drop=True)

# Bright Data remains untouched as the external test set.
bright_external_test_reviews = bright_recent.copy().reset_index(drop=True)

print("Kaggle supervised review rows:", len(kaggle_split_base))
print("Training rows:", len(kaggle_train_reviews))
print("Validation rows:", len(kaggle_validation_reviews))
print("Internal test rows:", len(kaggle_test_reviews))
print("Bright external test rows:", len(bright_external_test_reviews))


In [ ]:

# Verify that the split proportions are approximately 80 / 10 / 10.

split_summary = pd.DataFrame({
    "split": [
        "Kaggle train",
        "Kaggle validation",
        "Kaggle internal test",
        "Bright external test",
    ],
    "rows": [
        len(kaggle_train_reviews),
        len(kaggle_validation_reviews),
        len(kaggle_test_reviews),
        len(bright_external_test_reviews),
    ],
})

kaggle_total = len(kaggle_split_base)

split_summary["percent_of_kaggle"] = [
    100 * len(kaggle_train_reviews) / kaggle_total,
    100 * len(kaggle_validation_reviews) / kaggle_total,
    100 * len(kaggle_test_reviews) / kaggle_total,
    np.nan,
]

split_summary["percent_of_kaggle"] = (
    split_summary["percent_of_kaggle"].round(2)
)

display(split_summary)


In [ ]:

# Check whether sentiment proportions remain similar across the three Kaggle splits.

def sentiment_percent(df):
    return (
        df["sentiment_label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


split_sentiment_balance = pd.DataFrame({
    "Train_%": sentiment_percent(kaggle_train_reviews),
    "Validation_%": sentiment_percent(kaggle_validation_reviews),
    "Internal_test_%": sentiment_percent(kaggle_test_reviews),
    "Bright_external_test_%": sentiment_percent(bright_external_test_reviews),
}).fillna(0)

display(split_sentiment_balance)


In [ ]:

# Create training-ready sentiment and aspect tables from each review-level split.

kaggle_sentiment_train = make_sentiment_table(
    kaggle_train_reviews
)

kaggle_sentiment_validation = make_sentiment_table(
    kaggle_validation_reviews
)

kaggle_sentiment_test = make_sentiment_table(
    kaggle_test_reviews
)

kaggle_aspect_train = make_aspect_table(
    kaggle_train_reviews
)

kaggle_aspect_validation = make_aspect_table(
    kaggle_validation_reviews
)

kaggle_aspect_test = make_aspect_table(
    kaggle_test_reviews
)

# Bright Data remains external only.
bright_sentiment_external_test = make_sentiment_table(
    bright_external_test_reviews
)

bright_aspect_external_test = make_aspect_table(
    bright_external_test_reviews
)

training_table_summary = pd.DataFrame({
    "table": [
        "Kaggle sentiment train",
        "Kaggle sentiment validation",
        "Kaggle sentiment internal test",
        "Kaggle aspect train",
        "Kaggle aspect validation",
        "Kaggle aspect internal test",
        "Bright sentiment external test",
        "Bright aspect external test",
    ],
    "rows": [
        len(kaggle_sentiment_train),
        len(kaggle_sentiment_validation),
        len(kaggle_sentiment_test),
        len(kaggle_aspect_train),
        len(kaggle_aspect_validation),
        len(kaggle_aspect_test),
        len(bright_sentiment_external_test),
        len(bright_aspect_external_test),
    ],
})

display(training_table_summary)



## 13. Quality checks

Before saving, inspect:

- dataset sizes;
- date ranges;
- missing review text;
- missingness of rating variables;
- rating ranges;
- sentiment distributions;
- number of labeled rows per aspect.


In [ ]:

def quality_report(name, df):
    print("=" * 80)
    print(name)
    print("Rows:", len(df))

    if "date_review" in df.columns:
        print(
            "Date range:",
            df["date_review"].min(),
            "to",
            df["date_review"].max(),
        )

    if "review_text" in df.columns:
        print(
            "Missing review_text:",
            int(df["review_text"].isna().sum()),
        )

    rating_cols = [
        col
        for col in ["overall_rating"] + ASPECT_COLUMNS
        if col in df.columns
    ]

    if rating_cols:
        print("\nMissingness for ratings:")

        missing = (
            df[rating_cols]
            .isna()
            .mean()
            .mul(100)
            .round(2)
            .rename("missing_percent")
            .to_frame()
        )

        display(missing)

        print("Observed rating ranges:")

        for col in rating_cols:
            valid = df[col].dropna()

            if len(valid):
                print(
                    f"  {col}: "
                    f"{valid.min()} to {valid.max()}"
                )


quality_report(
    "Kaggle cleaned",
    kaggle_clean,
)

quality_report(
    "Bright Data recent cleaned",
    bright_recent,
)


In [ ]:

print("Kaggle training sentiment distribution")
display(
    kaggle_sentiment_train["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nKaggle validation sentiment distribution")
display(
    kaggle_sentiment_validation["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nKaggle internal-test sentiment distribution")
display(
    kaggle_sentiment_test["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nBright external-test sentiment distribution")
display(
    bright_sentiment_external_test["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nKaggle training aspect-label counts")
display(
    kaggle_aspect_train["aspect"]
    .value_counts()
    .to_frame("count")
)

print("\nBright external-test aspect-label counts")
display(
    bright_aspect_external_test["aspect"]
    .value_counts()
    .to_frame("count")
)


### Review ID verification (before exporting)

Every saved table must retain a non-empty `review_id` for linking model predictions
back to the original review in RAG. IDs must be **unique for review-level and
sentiment tables**; aspect tables may repeat an ID because one review can
produce multiple aspect rows. Kaggle training/validation/test IDs must be disjoint.


In [ ]:
# Verify that every exported table contains populated review IDs.
review_id_tables = {
    "kaggle_clean_reviews.csv": kaggle_clean,
    "brightdata_recent_clean_reviews.csv": bright_recent,
    "kaggle_reviews_train.csv": kaggle_train_reviews,
    "kaggle_reviews_validation.csv": kaggle_validation_reviews,
    "kaggle_reviews_test.csv": kaggle_test_reviews,
    "kaggle_sentiment_train.csv": kaggle_sentiment_train,
    "kaggle_sentiment_validation.csv": kaggle_sentiment_validation,
    "kaggle_sentiment_test.csv": kaggle_sentiment_test,
    "brightdata_sentiment_external_test.csv": bright_sentiment_external_test,
    "kaggle_aspect_train.csv": kaggle_aspect_train,
    "kaggle_aspect_validation.csv": kaggle_aspect_validation,
    "kaggle_aspect_test.csv": kaggle_aspect_test,
    "brightdata_aspect_external_test.csv": bright_aspect_external_test,
}

for filename, df in review_id_tables.items():
    assert "review_id" in df.columns, f"{filename}: review_id column missing"
    ids = df["review_id"].astype("string").str.strip()
    missing = ids.isna() | ids.eq("").fillna(True)
    assert not missing.any(), f"{filename}: {int(missing.sum())} empty review_id values"

    # Aspect tables have several rows per review, so duplicate IDs are expected.
    if "aspect_" not in filename and "_aspect" not in filename:
        assert ids.is_unique, f"{filename}: duplicate review_id values"

    print(f"OK: {filename}: {len(df):,} rows; IDs present")

train_ids = set(kaggle_train_reviews["review_id"])
validation_ids = set(kaggle_validation_reviews["review_id"])
test_ids = set(kaggle_test_reviews["review_id"])
assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)
print("OK: Kaggle train/validation/test review IDs are disjoint")


## 14. Save cleaned outputs

Files are saved to `data/processed/`.

### Cleaned review-level data
- `kaggle_clean_reviews.csv`
- `brightdata_recent_clean_reviews.csv`

### Kaggle review-level splits
- `kaggle_reviews_train.csv`
- `kaggle_reviews_validation.csv`
- `kaggle_reviews_test.csv`

### Sentiment tables
- `kaggle_sentiment_train.csv`
- `kaggle_sentiment_validation.csv`
- `kaggle_sentiment_test.csv`
- `brightdata_sentiment_external_test.csv`

### Aspect tables
- `kaggle_aspect_train.csv`
- `kaggle_aspect_validation.csv`
- `kaggle_aspect_test.csv`
- `brightdata_aspect_external_test.csv`

Bright Data remains separate from Kaggle and is used only as the recent external test set / later RAG source.


In [ ]:

OUTPUTS = {
    "kaggle_clean_reviews.csv":
        kaggle_clean,

    "brightdata_recent_clean_reviews.csv":
        bright_recent,

    "kaggle_reviews_train.csv":
        kaggle_train_reviews,

    "kaggle_reviews_validation.csv":
        kaggle_validation_reviews,

    "kaggle_reviews_test.csv":
        kaggle_test_reviews,

    "kaggle_sentiment_train.csv":
        kaggle_sentiment_train,

    "kaggle_sentiment_validation.csv":
        kaggle_sentiment_validation,

    "kaggle_sentiment_test.csv":
        kaggle_sentiment_test,

    "brightdata_sentiment_external_test.csv":
        bright_sentiment_external_test,

    "kaggle_aspect_train.csv":
        kaggle_aspect_train,

    "kaggle_aspect_validation.csv":
        kaggle_aspect_validation,

    "kaggle_aspect_test.csv":
        kaggle_aspect_test,

    "brightdata_aspect_external_test.csv":
        bright_aspect_external_test,
}

for filename, dataframe in OUTPUTS.items():
    path = PROCESSED_DIR / filename
    dataframe.to_csv(path, index=False)

    print(
        f"Saved {len(dataframe):,} rows -> {path}"
    )



## Final output

The cleaned data are now organized for the modeling stage.

Recommended use:

- **Kaggle 80% training set** → LoRA/model fine-tuning
- **Kaggle 10% validation set** → model selection and tuning
- **Kaggle 10% internal test set** → final same-source evaluation
- **Bright Data 2022+** → recent external evaluation and later RAG demonstration

The Bright Data records should remain untouched during training and model-selection decisions.

> The Kaggle training split is large. For resource-limited LoRA experiments, a smaller **stratified sample should be drawn only from the training split**, while validation and test sets remain unchanged.
